In [ ]:
# In this Jupyter notebook, we will embark on a journey to fine-tune a powerful language model using the Magicoder dataset. 
# Fine-tuning is a crucial step in adapting a pre-trained model to a specific task or domain, improving its performance and relevance.

# The **Magicoder dataset** is a specialized collection of instruction, input, and output triples. 
# It's designed for code generation tasks, containing approximately 1,000 examples. 
# Our goal is to leverage this dataset to fine-tune our chosen model, enabling it to generate high-quality Python code. 
# This dataset is unique because it's trained on a variety of programming languages such as Python, Java, C++, etc. 
# However, for our specific application, we will meticulously filter this dataset to focus exclusively on Python, 
# thereby restricting our fine-tuned model to generate only Python code. This ensures a specialized and efficient model. 
# Furthermore, the model will be designed to gracefully handle requests for other programming languages by providing 
# appropriate warnings and reminding the user of its Python-only expertise.

# Here's a detailed breakdown of the steps involved in fine-tuning a language model with any dataset:
# 1. **Data Identification**: The very first step is to pinpoint the specific dataset that aligns with your fine-tuning objectives. 
#    The quality and relevance of the dataset directly impact the performance of the fine-tuned model.
# 2. **Dataset Understanding**: A deep dive into the dataset's structure, format, and content is paramount. 
#    This understanding is critical because the dataset must be transformed into a format that the model can readily process. 
#    This often involves specific tokenization or prompt formatting.
# 3. **Model and Tokenizer Selection**: Choose a suitable pre-trained base model that will be fine-tuned, 
#    along with its corresponding tokenizer. The tokenizer is responsible for converting human-readable text into numerical 
#    tokens that the model can understand, and vice-versa.
# 4. **Prompting Template Comprehension**: Models, especially large language models (LLMs), often expect input in a specific 
#    `prompting template` format. Understanding and converting your dataset examples into this format is a vital prerequisite 
#     for effective fine-tuning.
# 5. **Data Transformation Function**: Develop a function that systematically converts raw dataset examples into the model's 
#    expected input format, adhering to the identified prompting template.
# 6. **Dataset Loading and Conversion**: Load the chosen dataset into memory and apply the transformation function to prepare it 
#    for training.
# 7. **Model Configuration**: Identify and set the various parameters and properties that govern the fine-tuning process. 
#    Key configurations include:
#    7.1 **QLoRA or LoRA Configuration**: Low-Rank Adaptation (LoRA) and Quantized LoRA (QLoRA) are parameter-efficient fine-tuning 
#        techniques that significantly reduce computational cost and memory usage. These configurations define how these techniques 
#        are applied.
#    7.2 **Model Specific Configuration**: This includes essential details like the base model name, the tokenizer name, and 
#        the desired name for your fine-tuned output model.
#    7.3 **Quantization Configuration**: If using QLoRA, this involves setting parameters for quantization (e.g., 4-bit, 8-bit), 
#        which further reduces model size and speeds up inference by representing model weights with lower precision.
#    7.4 **Training Configuration**: These parameters control the training loop itself, such as the number of epochs 
#        (how many times the entire dataset is passed through the model), 
#        batch size (number of samples processed before updating model weights), 
#        and the learning rate (step size for weight updates).
#    7.5 **SFT Parameters**: Specific parameters for Supervised Fine-Tuning (SFT), including the prompt template used, 
#         overall model details, QLoRA parameters, bits and bytes settings, and which parts of the model are trainable.
# 8. **Model Training**: Execute the training process where the model learns from the prepared dataset, adjusting its internal 
#    weights to better perform the target task.
# 9. **Model Evaluation**: After training, rigorously evaluate the model's performance on a separate validation set to ensure 
#    it generalizes well and meets performance benchmarks.
# 10. **Model and Tokenizer Saving**: Persist the fine-tuned model and its tokenizer to disk so they can be reloaded and 
#     used later without retraining.
# 11. **Model Merging (Optional but Recommended)**: If using LoRA/QLoRA, merge the adapters (the small, fine-tuned layers) 
#     back into the base model to create a unified model with the original base model. This creates a single, self-contained 
#     model that is easier to deploy and share.
# 12. **Memory Cleanup**: Release computational resources (GPU memory, CPU memory) by deleting model and tokenizer objects from memory. 
#     This is crucial for subsequent operations or to free up resources.
# 13. **Model Reloading**: Load the newly merged (or fine-tuned) model and its tokenizer from disk to verify its 
#     integrity and prepare for inference.
# 14. **Final Validation**: Test the fine-tuned model with new, unseen user prompts to confirm its ability to generate accurate 
#     and relevant responses in real-world scenarios.

In [ ]:
# This section focuses on authenticating with the Hugging Face Hub, a central repository for pre-trained models, datasets, and more. 
# It also demonstrates how to interact with the Hub programmatically.

# The `notebook_login()` function from `huggingface_hub` allows you to securely log in to your Hugging Face account 
# directly from a Jupyter or Colab notebook. This grants you access to private models, allows you to upload your own models, 
# and increases API rate limits.
from huggingface_hub import notebook_login
import os

notebook_login()

# After successful login, you can use the `list_models` function to programmatically browse the vast collection of models 
# available on the Hugging Face Hub. Here, we're demonstrating how to list models that might be relevant, for example, 
# those whose names start with "llama".
from huggingface_hub import list_models

hf_models = list_models()

# The `HF_TOKEN` environment variable is crucial for authenticating your requests to the Hugging Face Hub, 
# especially when working with private models or uploading artifacts. This block checks if the token is properly set in 
# the Colab's 'Secrets' panel, which is the recommended secure way to store sensitive information. If it's not found, 
# it provides clear instructions on how to set it up, or how to set it programmatically for 
# quick testing (though the latter is less secure for persistent use). This ensures your operations with Hugging Face are 
# authenticated.
if "HF_TOKEN" not in os.environ:
    print("Warning: The 'HF_TOKEN' environment variable is not set.")
    print("Please set it in Colab's 'Secrets' panel (key: HF_TOKEN, value: your_huggingface_token). This is the recommended secure method.")
    print("Alternatively, for temporary testing, you can set it directly in a code cell: os.environ['HF_TOKEN'] = 'your_token_here' (Note: This is less secure for production environments).")
    # Assign a placeholder to avoid KeyError if the print statement is the only goal
    hf_token = "placeholder_token_not_set"
else:
    hf_token = os.environ["HF_TOKEN"]
    print(f"Token from environment: {hf_token[:20]}...") # Prints a snippet of the token for verification without revealing the full token.

# This line demonstrates how to retrieve the Hugging Face token specifically from Colab's `userdata` secrets manager, 
# providing another way to access securely stored credentials.
from google.colab import userdata
# Truncate the displayed token to only show a few characters
print(f"Token from userdata: {userdata.get('HF_TOKEN')[:12]}...")

In [ ]:
# When working in Google Colab, external libraries often need to be installed. These commented-out lines indicate commands 
# that were previously run (or might need to be run) to install necessary packages.
# `!pip install trl`: This command installs the `trl` library, which stands for "Transformer Reinforcement Learning." 
#                     This library is particularly useful for fine-tuning large language models using techniques like 
#                     Supervised Fine-Tuning (SFT) and Reinforcement Learning from Human Feedback (RLHF).
# `!pip install bitsandbytes`: This command installs the `bitsandbytes` library, which provides efficient implementations 
#                              of 8-bit and 4-bit quantization. Quantization is a technique used to reduce the memory footprint 
#                              and speed up the inference of large models by storing their weights at lower precision 
#                              (e.g., 4-bit integers instead of 32-bit floats).

In [ ]:
# This cell is dedicated to importing all the necessary Python packages and modules that will be used throughout the 
# fine-tuning process. Each import serves a specific purpose:
import torch # PyTorch is an open-source deep learning framework widely used for building and training neural networks. 
             # It provides powerful tensor computation (like NumPy with GPU acceleration) and a comprehensive ecosystem 
             # for deep learning research and development.

# The `datasets` library from Hugging Face is an incredibly useful tool for easily loading, processing, and sharing datasets. 
# It handles various data formats and provides functionalities for efficient data manipulation.
from datasets import load_dataset

# The `transformers` library, also from Hugging Face, is a state-of-the-art library for natural language processing (NLP), 
# providing thousands of pre-trained models (like BERT, GPT, T5) to perform tasks on texts. 
# It includes:
# `AutoModelForCausalLM`: This class is used to load pre-trained models specifically designed for causal language modeling tasks 
# (i.e., predicting the next token in a sequence). It automatically infers the correct model architecture from the model name.
# `AutoTokenizer`: This class is a factory for loading the correct tokenizer associated with a given pre-trained model. 
# Tokenizers are essential for converting raw text into numerical input IDs that the model can process, and vice-versa.
from transformers import AutoTokenizer, AutoModelForCausalLM

# Quantization is a crucial technique for deploying large models efficiently. It involves reducing the precision of the 
# model's weights and activations (e.g., from 32-bit floating-point numbers to 4-bit integers) to decrease memory usage 
# and accelerate computation without significant performance loss.
# `BitsAndBytesConfig`: This class provides a convenient way to configure the quantization settings, such as the number of bits 
# for quantization (e.g., 4-bit), the type of quantization (e.g., `nf4` for NormalFloat4), and whether to use nested 
# quantization or specific compute data types.
from transformers import BitsAndBytesConfig

# The `trl` (Transformer Reinforcement Learning) library from Hugging Face simplifies the process of fine-tuning 
# large language models using various methods, including Supervised Fine-Tuning (SFT).
# `SFTTrainer`: This class is a specialized `Trainer` designed for Supervised Fine-Tuning. 
# It streamlines the training loop, handles data preparation, and integrates seamlessly with `transformers` models and 
# `peft` (Parameter-Efficient Fine-Tuning) configurations.

In [ ]:

# Here, we're loading the specific dataset that will be used for fine-tuning our model. 
# The `Magicoder-OSS-Instruct-75K` dataset is an excellent choice for code generation tasks.

# **About Magicoder-OSS-Instruct-75K:** This dataset is meticulously constructed using an execution-based approach. 
# This means it generates clean, diverse coding problems alongside structural script traces across numerous programming languages. 
# Its design makes it particularly effective for teaching a model intricate syntax rules and programming logic, 
# rather than just generating generic boilerplate code. It provides a rich set of examples for the model to learn from.

dataset_name = "ise-uiuc/Magicoder-OSS-Instruct-75K"
dataset = load_dataset(dataset_name, split="train") # We load the 'train' split of the dataset, which contains the examples our model will learn from.

In [ ]:
# This cell is dedicated to performing an initial exploration of the loaded dataset. It's crucial to understand the 
# structure, content, and characteristics of the data before proceeding with fine-tuning.

# `print(dataset)`: This command provides a high-level overview of the dataset object. It typically shows information 
# such as the dataset's name, the number of rows (examples), and the column names along with their data types. 
# This helps in quickly grasping the dataset's schema.
print(dataset)

# `for i in range(3): print(dataset[i])`: This loop iterates through the first three examples (rows) of the dataset and 
# prints their content. This allows for a detailed inspection of individual data points, helping to understand the 
# actual format of the 'instruction', 'input', and 'output' fields, or 'problem' and 'solution' fields, as seen in this dataset. 
# It's a way to visually confirm what the data looks like.
# Look at the first 3 examples
for i in range(3):
    print(dataset[i])

# `for i in range(len(dataset) - 3, len(dataset)): print(dataset[i])`: Similarly, this loop prints the last three examples of 
# the dataset. Checking the tail of the dataset can sometimes reveal different patterns or data quality issues compared 
# to the head, ensuring a comprehensive initial review.
# Look at the last 3 examples
for i in range(len(dataset) - 3, len(dataset)):
    print(dataset[i])

In [ ]:
# As this data set is trained for multiple programming languages, we will filter the data set to only Python code.
# I am fine tuning this data set only to generate Python code and restric the model to generate only Python code.
dataset = dataset.filter(lambda example: example["lang"] == "python")
print(dataset)

# Look at the first 3 examples
for i in range(3):
    print(dataset[i])

# print probem and solution
for i in range(3):
    print(f"Problem: {dataset[i]['problem']}\nSolution: {dataset[i]['solution']}\n") # print(dataset[i]["problem"])


In [ ]:
# This cell defines a comprehensive set of parameters and configurations essential for the fine-tuning process. 
# These parameters cover the base model details, LoRA/QLoRA specific settings, quantization configurations, 
# and the overall training arguments.

# **Model Configuration Parameters (`model_config`)**
# This dictionary holds information about the base model, tokenizer, and the names for the new fine-tuned and merged models.
# `model_name_or_path`: Specifies the identifier of the pre-trained model on the Hugging Face Hub (e.g., "meta-llama/Llama-2-7b-hf"). 
#                       This is the model that will be loaded and fine-tuned.
# `tokenizer_name_or_path`: Specifies the identifier for the tokenizer corresponding to the `model_name_or_path`. 
#                           Tokenizers are crucial for converting text into a format the model understands.
# `device_map`: Determines how the model's layers are distributed across available hardware (e.g., GPUs). 
#               Setting it to "auto" lets the `transformers` library automatically assign layers to optimize memory usage.
# `new_model_name`: The desired name for the LoRA adapters or the intermediate fine-tuned model after training. 
#                   This name is used when saving the PEFT (Parameter-Efficient Fine-Tuning) model.
# `merged_model_name`: The desired name for the final, merged model. After fine-tuning with LoRA, the adapters are merged 
#                      back into the base model to create a single, deployable entity.
# `data_set_name`: The name or path of the dataset used for fine-tuning.
model_config = {
    "model_name_or_path": "meta-llama/Llama-2-7b-chat-hf",
    "tokenizer_name_or_path": "meta-llama/Llama-2-7b-chat-hf",
   ## "use_auth_token": True,
   ## "trust_remote_code": True,
    "device_map": "auto",
    "new_model_name": "llama-2-7b-finetuned-magicoder",
    "merged_model_name": "llama-2-7b-finetuned-magicoder-merged",
    "data_set_name": dataset_name,
}

# **QLoRA Parameters (`QLoRA_parameters`)**
# QLoRA (Quantized Low-Rank Adaptation) is an efficient fine-tuning method that uses 4-bit quantization. These parameters 
# configure the LoRA part of QLoRA.
# `lora_r`: The rank of the update matrices in LoRA. A higher rank allows for more expressiveness but increases the number 
#           of trainable parameters.
# `lora_alpha`: A scaling factor that adjusts the magnitude of the LoRA updates.
# `lora_dropout`: The dropout probability applied to the LoRA layers for regularization, helping to prevent overfitting.
# `lora_target_modules`: A list of module names (e.g., `q_proj`, `v_proj` which correspond to query and value projection layers 
#                        in attention mechanisms) within the base model where LoRA adapters will be injected.
# `bias`: Defines how bias terms are handled in LoRA. 'none' means bias terms are not included in the LoRA adaptation.
# `task_type`: Specifies the type of NLP task. `CAUSAL_LM` (Causal Language Modeling) is used for models that 
#              predict the next token in a sequence.
QLoRA_paramters = {
    "lora_r": 64,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "lora_target_modules": ["q_proj", "v_proj"],
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

# **BitsAndBytes Quantization Parameters (`bits_and_bytes`)**
# This dictionary configures the 4-bit quantization settings provided by the `bitsandbytes` library, which is integral to QLoRA.
# `load_in_4bit`: If `True`, the base model weights are loaded in 4-bit precision.
# `bnb_4bit_quant_type`: The specific type of 4-bit quantization. 'nf4' (NormalFloat4) is a common and effective choice for LLMs.
# `use_nested_quant`: If `True`, applies nested quantization, quantizing the quantization constants themselves 
#                     for further memory savings.
# `bnb_4bit_compute_dtype`: The `torch.dtype` to use for computation during 4-bit operations 
#                           (e.g., `torch.float16` for faster computation).
# `use_4bit`: A general flag to indicate the use of 4-bit quantization.
# `bnb_4bit_use_double_quant`: If `True`, enables double quantization.
bits_and_bytes = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "use_nested_quant": False,
    "bnb_4bit_compute_dtype": torch.float16,
    "use_4bit": True,
    "bnb_4bit_use_double_quant": True,

}

# **Training Parameters (`training_parameters`)**
# This dictionary encapsulates various hyperparameters and settings that control the training loop itself for 
# Supervised Fine-Tuning (SFT).
# `num_train_epochs`: The total number of times the training algorithm will iterate over the entire dataset.
# `output_dir`: The directory where the fine-tuned model checkpoints and training artifacts will be saved.
# `learning_rate`: The initial learning rate for the optimizer. This determines the step size for updating model weights.
# `per_device_train_batch_size`: The number of training examples processed per GPU (or other device) in a single batch.
# `gradient_accumulation_steps`: The number of steps to accumulate gradients before performing a single optimization step. 
#                                This allows for effectively larger batch sizes than `per_device_train_batch_size`.
# `save_steps`: How often (in steps) to save a model checkpoint during training.
# `logging_steps`: How often (in steps) to log training metrics (e.g., loss, learning rate) to the console and/or TensorBoard.
# `warmup_ratio`: The proportion of training steps during which the learning rate will gradually increase from zero to 
#                 its initial value.
# `lr_scheduler_type`: The strategy used to adjust the learning rate over the course of training 
#                      (e.g., 'cosine' for a cosine decay schedule).
# `weight_decay`: A regularization parameter that penalizes large weights, helping to prevent overfitting.
# `max_grad_norm`: The maximum norm of the gradients. Gradients are clipped to this value to prevent exploding gradients, 
#                  especially in deep neural networks.
# `fp16`: If `True`, enables mixed-precision training using 16-bit floating-point numbers.
# `bf16`: If `True`, enables mixed-precision training using bfloat16 floating-point numbers 
#         (a format often preferred on TPUs and newer GPUs).
# `bf16_full_eval`: If `True`, performs evaluation in bfloat16 precision.
# `optim`: The optimizer to use for training (e.g., 'paged_adamw_32bit' is often used with QLoRA).
# `logging_dir`: The directory where training logs (for TensorBoard) will be stored.
# `report_to`: Specifies the reporting tools for logging (e.g., 'tensorboard').
# `max_steps`: The maximum number of training steps to perform. If set, it overrides `num_train_epochs`.
training_parameters = {
    "num_train_epochs": 3,
    "output_dir": "./results",
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "save_steps": 100,
    "logging_steps": 10,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.1,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "bf16_full_eval": False,
    "optim": "paged_adamw_32bit",
    "logging_dir": "logs",
    "report_to": "tensorboard",
    "max_steps": 10
}

# **Consolidated Fine-Tuning Parameters (`fine_tuning_parameters`)**
# This single dictionary aggregates all the individual configuration dictionaries for easy access and management.
# `model_details`: Contains the base model and tokenizer information.
# `QLoRA_parameters`: Contains the LoRA specific settings.
# `bits_and_bytes`: Contains the 4-bit quantization settings.
# `training_parameters`: Contains the training hyperparameters.
# `max_seq_length`: The maximum length of the input sequences (in tokens) that the model will process. 
#                   Longer sequences are truncated, shorter ones are padded.
# `packing`: A boolean flag. If `True`, it enables data packing, where multiple short sequences are concatenated to form longer ones, 
#            making more efficient use of GPU memory and speeding up training.
fine_tuning_parameters = {
    "model_details": model_config,
    "QLoRA_parameters": QLoRA_paramters,
    "bits_and_bytes": bits_and_bytes,
    "training_parameters": training_parameters,
    "max_seq_length": None,
    "packing" : False,
}

In [ ]:
# This cell demonstrates how to obtain and use the specific prompt template expected by the chosen large language model (LLM), 
# in this case, a Llama-2-based model. LLMs are often trained with a particular input format, and adhering to this format during 
# fine-tuning and inference is crucial for optimal performance.

# First, we import `AutoTokenizer` and `AutoModelForCausalLM` from the `transformers` library. 
# While `AutoModelForCausalLM` is not directly used for prompt formatting here, it's often imported together with the tokenizer.
from transformers import AutoTokenizer, AutoModelForCausalLM

# **Tokenizer Loading and Configuration**
# `tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])`: The tokenizer corresponding to our base model 
#              is loaded. This tokenizer is responsible for converting raw text into numerical tokens (input IDs) that the 
#              model can understand.
# `tokenizer.pad_token = tokenizer.eos_token`: For batch processing, sequences often need to be padded to a uniform length. 
#                        We set the padding token to be the 'end-of-sequence' token (`eos_token`). This is a common practice 
#                        for causal language models.
# `tokenizer.padding_side = "right"`: Specifies that padding should be added to the right side of the sequence. 
#                           This is important because some models are more sensitive to where padding is applied.
tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])
tokenizer.pad_token = tokenizer.eos_token # End of the prompt template
tokenizer.padding_side = "right" # right padding the prompt template

# **Example Conversation for Template Demonstration**
# This `messages` list simulates a multi-turn conversation, which is typical for chat-based LLMs. 
# Each dictionary represents a turn, with a `role` (e.g., 'user', 'assistant') and `content`.
messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is the capital of Italy?"},
    {"role": "assistant", "content": "The capital of Italy is Rome."},
    {"role": "user", "content": "What is the capital of Germany?"},
    {"role": "assistant", "content": "The capital of Germany is Berlin."},
    {"role": "user", "content": "What is the capital of Spain?"},
    {"role": "assistant", "content": "The capital of Spain is Madrid."},
    {"role": "user", "content": "What is the capital of Portugal?"}
]

# **Formatting the Prompt Template**
# `formatted_prompt = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)`: 
#                     This is the core function call. It takes the list of `messages` and converts them into a single string 
#                     formatted according to the model's expected chat template.
# `tokenizer=False`: This argument tells the function to return a string rather than tokenized IDs.
# `add_generation_prompt=True`: This adds a special token at the end of the prompt that signals to the model that it should 
#                        start generating a response.
# `print(formatted_prompt)`: Displays the raw string output of the formatted prompt, showing how the messages are 
#                            structured with special tokens (e.g., `<s>`, `<INST>`, `<<sys>>`, `<</sys>>`).
# `print(tokenizer.decode(formatted_prompt["input_ids"]))`: If `tokenizer=True` were used, this would decode the token IDs 
#                          back into a human-readable string. However, since `tokenizer=False` is used, `formatted_prompt` 
#                          is already a string.
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_prompt)
# The `formatted_prompt` variable holds the string representation, so `formatted_prompt["input_ids"]` would lead to an error.
# To show how the tokenizer decodes, we would typically tokenize the `formatted_prompt` string first if we wanted to 
# demonstrate both tokenization and decoding.
# Example if `tokenizer=True` was used:
# tokenized_output = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
# print(tokenizer.decode(tokenized_output))
# For this specific case, as `tokenize=False` is used, `formatted_prompt` is already a string. 
# If we wanted to decode, we'd tokenize it first:
# print(tokenizer.decode(tokenizer.encode(formatted_prompt)))
# However, as the goal is to show the *formatted string*, the direct print is sufficient.


In [ ]:
# This cell is commented out, but it represents a common practice when defining a custom prompt template for fine-tuning.
# The structure shown here is a typical prompt template for Llama-2-based models, which use special tokens 
# (`<s>`, `</s>`, `<INST>`, `</INST>`, `<<sys>>`, `<</sys>>`, `<<user>>`, `<</user>>`) to delineate different parts 
# of a conversation: the beginning/end of a sequence, instructions, system messages, user inputs, and model answers.
# Defining `model_config["prompt_template"]` in this manner allows for a consistent and structured way to format input 
# prompts during fine-tuning, ensuring the model receives input in the format it was designed to understand, 
# which is crucial for optimal performance.
# The `system_message` field provides overarching instructions or persona to the model, while `user_message` 
# is the actual query, and `model_answer` is the expected response to be learned.

In [ ]:
# This cell defines a crucial data transformation function, `convert_to_prompt_format`, which prepares the raw dataset examples 
# into the specific chat-based prompt format expected by the Llama-2 model. This function ensures consistency 
# between the training data and the model's architecture.

# **`convert_to_prompt_format(example)` Function**
# This function takes a single example (row) from the dataset as input, typically containing 'problem' and 'solution' fields.
def convert_to_prompt_format(example):
    # **System Message Definition**
    # `system_msg`: This string defines the 'system' role in the conversation, setting the persona and constraints for the model. 
    # In this case, it instructs the model to act as a Python automation assistant and explicitly states that it only supports 
    # Python code, providing a refusal message for other languages.
    system_msg = (
        "You are a Python automation assistant. You only support and generate Python code. "
        "If a user asks you for code in any other language (like Java, C++, or JavaScript), "
        "you must explicitly refuse and remind them of your Python specialization boundaries."
    )

    # **Message Structure for `apply_chat_template`**
    # `messages`: This list structures the conversation turns. It follows the standard `role` (e.g., 'system', 'user', 'assistant') 
    #             and `content` format expected by `tokenizer.apply_chat_template`.
    # - The first entry is the `system_msg`.
    # - The second is the user's `problem`.
    # - The third is the assistant's `solution` (the expected output).
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]}
    ]

    # **Formatting with Tokenizer**
    # `formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)`: 
    # This line uses the previously loaded tokenizer to convert the `messages` list into a single string formatted according 
    # to the model's chat template. `tokenize=False` ensures that a string is returned, not token IDs.
    # `return {"text": formatted_text}`: The function returns a dictionary where the key 'text' holds the fully formatted prompt 
    # string. This is the format `SFTTrainer` expects for its `dataset_text_field`.
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted_text}

# **Applying the Transformation to the Dataset**
# `print(dataset.column_names)`: Prints the original column names of the dataset before transformation. 
# This helps confirm the starting schema.
print(dataset.column_names)
# `dataset = dataset.map(convert_to_prompt_format, remove_columns=dataset.column_names)`: 
# This is a powerful operation from the `datasets` library. It applies the `convert_to_prompt_format` function to every example 
# in the `dataset`. The `remove_columns=dataset.column_names` argument instructs the method to remove all original columns 
# and only keep the new 'text' column generated by our function. This results in a dataset where each example is a dictionary 
# with a single 'text' key containing the formatted prompt.
dataset = dataset.map(convert_to_prompt_format, remove_columns=dataset.column_names)
# `print(dataset)`: Prints the updated dataset information, showing that it now contains only the 'text' column, 
# ready for fine-tuning.
print(dataset)

In [ ]:
# When running code in Google Colab, it's common to encounter missing library errors if the required packages haven't been 
# installed in the current environment.
# `!pip install peft`: This command installs the `peft` (Parameter-Efficient Fine-Tuning) library, 
#                      which is essential for implementing techniques like LoRA (Low-Rank Adaptation). 
#                      PEFT significantly reduces the number of trainable parameters during fine-tuning, making it much 
#                      more computationally efficient and memory-friendly, especially for large models.

# **Quantization Configuration**
# This dictionary defines the parameters for 4-bit quantization, a technique used to compress the model by representing its 
# weights with fewer bits. This reduces memory usage and speeds up inference.
# `load_in_4bit`: A boolean flag that, when set to `True`, instructs the model to load its weights in 4-bit precision.
# `bnb_4bit_quant_type`: Specifies the type of 4-bit quantization. 'nf4' (NormalFloat4) is a data type 
#                        specifically optimized for transformers.
# `bnb_4bit_compute_dtype`: Defines the data type used for computation during operations. `torch.float16` 
#                           (half-precision floating-point) is commonly used to balance speed and numerical stability.
# `bnb_4bit_use_double_quant`: A boolean flag for whether to use double quantization, 
#                              which quantizes the quantization constants, further reducing memory.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=bits_and_bytes["load_in_4bit"],
    bnb_4bit_quant_type=bits_and_bytes["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=bits_and_bytes["bnb_4bit_compute_dtype"],
    bnb_4bit_use_double_quant=bits_and_bytes["bnb_4bit_use_double_quant"]
)

from peft import LoraConfig
import peft
# **LoRA (Low-Rank Adaptation) Configuration**
# LoRA is a parameter-efficient fine-tuning method where low-rank matrices are injected into the transformer layers, 
# and only these new matrices are trained. This dramatically reduces the number of trainable parameters while achieving 
# comparable performance to full fine-tuning.
# `r`: This parameter defines the rank of the update matrices used in LoRA. A higher rank means more parameters to train, 
#      potentially leading to better performance but also higher computational cost.
# `lora_alpha`: This is a scaling factor for the LoRA updates. It controls the magnitude of the updates to the original weights.
# `target_modules`: This specifies which modules (layers) within the pre-trained model will have LoRA applied to them. 
#                   Common targets for language models include the query (`q_proj`) and value (`v_proj`) projection layers 
#                   in the attention mechanism, as these are often good places for efficient adaptation.
# `lora_dropout`: This is the dropout probability applied to the LoRA layers. Dropout is a regularization technique that helps 
#                 prevent overfitting by randomly setting a fraction of layer outputs to zero during training.
# `bias`: Specifies how bias terms are handled. 'none' means no bias terms are trained with LoRA, which is a common and 
#         effective choice.
# `task_type`: This parameter indicates the type of task the model is being fine-tuned for. 
#              `CAUSAL_LM` stands for Causal Language Modeling, where the model predicts the next token in a sequence.
peft_config = LoraConfig(
    r=fine_tuning_parameters["QLoRA_parameters"]["lora_r"],
    lora_alpha=fine_tuning_parameters["QLoRA_parameters"]["lora_alpha"],
    target_modules=fine_tuning_parameters["QLoRA_parameters"]["lora_target_modules"],
    lora_dropout=fine_tuning_parameters["QLoRA_parameters"]["lora_dropout"],
    bias=fine_tuning_parameters["QLoRA_parameters"]["bias"],
    task_type=fine_tuning_parameters["QLoRA_parameters"]["task_type"]
)

In [ ]:
# Similar to previous installation notes, this commented-out line indicates that `bitsandbytes` might need an upgrade.
# `!pip install -U bitsandbytes`: The `-U` flag ensures that the package is upgraded to its latest version if it's already installed. 
# An upgraded `bitsandbytes` library can bring performance improvements, bug fixes, or compatibility with newer versions of 
# `transformers` or `torch`, especially when dealing with advanced quantization features like 4-bit loading for 
# large language models.

In [ ]:
# Load pre-trained model with LoRA configuration
model = AutoModelForCausalLM.from_pretrained(model_config["model_name_or_path"],
                                             quantization_config=quantization_config,
                                             device_map=model_config["device_map"])

# Prevent the model from using the cache
model.config.use_cache = False

# Set the number of threads to use
model.config.pretraining_tp = 1

# Load tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])

# Set pad token and padding side
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [ ]:
# This cell sets up and initiates the Supervised Fine-Tuning (SFT) process using the `trl` library's `SFTTrainer`. 
# This is where the model will learn from the prepared dataset.

# First, we import the necessary classes from `transformers` and `trl`.
from transformers import TrainingArguments, Trainer
from trl import SFTConfig, SFTTrainer

# **Supervised Fine-Tuning Configuration (`training_args`)**
# `SFTConfig`: This class is a specialized configuration object for `SFTTrainer`, 
#              inheriting from `TrainingArguments` from `transformers`. It encapsulates all the training hyperparameters 
#              defined earlier in `training_parameters`.
# `output_dir`: The directory to save model checkpoints and other training outputs.
# `per_device_train_batch_size`: The batch size per GPU.
# `gradient_accumulation_steps`: Number of steps to accumulate gradients before an optimization step, 
#                                effectively increasing batch size.
# `learning_rate`: The initial learning rate.
# `num_train_epochs`: The number of full passes over the training dataset.
# `lr_scheduler_type`: The learning rate scheduling strategy (e.g., 'cosine').
# `weight_decay`: L2 regularization parameter.
# `max_grad_norm`: Gradient clipping threshold.
# `fp16`, `bf16`: Flags for mixed-precision training.
# `optim`: The optimizer to use.
# `logging_steps`: How frequently to log training information.
# `max_steps`: Maximum number of training steps (overrides epochs if set).
# `gradient_checkpointing=True`: A memory-saving technique that recomputes activations during the backward pass instead of 
#                                storing them, reducing memory footprint at the cost of slight computational overhead. 
#                                This is often crucial for training large models.
# `dataset_text_field="text"`: This tells the `SFTTrainer` which column in our `dataset` 
#                              (which we prepared using `convert_to_prompt_format`) contains the text that should 
#                              be used for training.
training_args = SFTConfig(
    output_dir=training_parameters["output_dir"],
    per_device_train_batch_size=training_parameters["per_device_train_batch_size"],
    gradient_accumulation_steps=training_parameters["gradient_accumulation_steps"],
    learning_rate=training_parameters["learning_rate"],
    num_train_epochs=training_parameters["num_train_epochs"],
    lr_scheduler_type=training_parameters["lr_scheduler_type"],
    weight_decay=training_parameters["weight_decay"],
    max_grad_norm=training_parameters["max_grad_norm"],
    fp16=False,
    bf16=True,
    optim=training_parameters["optim"],
    logging_steps=training_parameters["logging_steps"],
    max_steps=training_parameters["max_steps"],
    gradient_checkpointing=True, # Added gradient_checkpointing, it's a boolean value that tells the model to use gradient checkpointing
    dataset_text_field="text"  # see what you have defined in convert_to_prompt_format
)

# **Creating and Running the `SFTTrainer`**
# `trainer = SFTTrainer(...)`: An instance of `SFTTrainer` is created. It takes the following key arguments:
# - `model`: The pre-trained `AutoModelForCausalLM` instance (loaded with quantization).
# - `train_dataset`: Our prepared dataset (with the 'text' field).
# - `peft_config`: The `LoraConfig` object, which tells the trainer to use LoRA for parameter-efficient fine-tuning.
# - `args`: The `SFTConfig` object containing all the training arguments.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,

    args=training_args
)
# `trainer.train()`: This method starts the fine-tuning process. The trainer will iterate through the dataset, 
#                    perform forward and backward passes, update model weights according to the specified optimizer 
#                    and scheduler, and log metrics as configured.
trainer.train()

In [ ]:
# After the fine-tuning process is complete, it's essential to save the trained model and its tokenizer so they can be 
# reloaded and used later without needing to retrain. This cell handles the saving of the PEFT (LoRA) adapters and the tokenizer.

# `trainer.model.save_pretrained(model_config["new_model_name"])`: This line saves the fine-tuned model's PEFT adapters. 
# When using LoRA, the `trainer.model` object actually holds the base model with the LoRA adapters layered on top. 
# This `save_pretrained` method saves only the *adapter weights* to the directory specified by `model_config["new_model_name"]`.
# It does not save the entire base model, only the small, efficient LoRA layers that were trained.
trainer.model.save_pretrained(model_config["new_model_name"])

# `tokenizer.save_pretrained(model_config["new_model_name"])`: The tokenizer, which is also a crucial component 
# for model inference (converting text to token IDs and back), is saved to the same directory as the PEFT adapters. 
# This ensures that when the fine-tuned model is loaded later, the correct tokenizer is associated with it.
tokenizer.save_pretrained(model_config["new_model_name"])

In [ ]:
# This cell is critical for memory management, especially in environments like Google Colab where GPU memory can be limited. 
# After training, the model, tokenizer, and trainer objects can still occupy significant GPU (and CPU) memory. 
# Clearing these resources is necessary before attempting to load a new model or perform other memory-intensive operations.

# `import gc`: Imports the `gc` module, which provides an interface to the garbage collector. 
# This allows for manual control over memory cleanup.
gc.collect()

# `del model`: Explicitly deletes the `model` object from memory. 
# This is particularly important for large language models that consume a lot of GPU VRAM.
del model

# `del tokenizer`: Deletes the `tokenizer` object. 
# While typically smaller than the model, it's good practice to clear it as well.
del tokenizer

# `del trainer`: Deletes the `trainer` object, which can hold references to the model, optimizer, and other training states, 
# thus freeing up associated memory.
del trainer

# `torch.cuda.empty_cache()`: This PyTorch specific function clears any unused cached memory from the GPU. 
# Even after deleting objects, some memory might remain allocated in PyTorch's cache, and this command helps to 
# release it back to the system, making it available for subsequent operations.
torch.cuda.empty_cache()

In [ ]:
# This section handles the installation of `torchao` and the crucial step of merging the fine-tuned LoRA adapters 
# back into the base model. This merging process creates a single, unified model that is easier to deploy and use without 
# needing to manage separate adapter weights.

# `!pip install --upgrade torchao`: This command installs or upgrades the `torchao` library. 
#                 `torchao` is a PyTorch library that offers tools for advanced quantization and optimization for large models, 
#                  which can be critical for efficient deployment of fine-tuned LLMs.

# **Merging the Base Model and Tokenizer with the Fine-Tuned Model and Tokenizer**
# After fine-tuning with PEFT methods like LoRA, you have a base model and a smaller set of adapter weights. 
# To create a standalone, deployable model, these adapters need to be merged back into the base model.
from peft import PeftModel

# First, the original pre-trained base model is reloaded. It's important to load it with the same quantization 
# configuration (if any) and data type (`torch.float16`) to ensure compatibility with the fine-tuned adapters. 
# `device_map="auto"` automatically distributes the model across available devices (e.g., GPUs).
base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_config["model_name_or_path"],
    quantization_config=quantization_config, # This ensures the base model is loaded with the same quantization settings as used during training.
    torch_dtype=torch.float16,
    device_map="auto"
)

# The `PeftModel.from_pretrained` method loads the fine-tuned adapter weights and wraps the base model. 
# This effectively 'attaches' the learned LoRA layers to the base model.
peft_model = PeftModel.from_pretrained(base_model_reload, model_config["new_model_name"])

# The `merge_and_unload()` method is the core of this step. It mathematically fuses the LoRA adapter weights with the 
# original base model's weights, creating a new, singular model. After this operation, the `peft_model` object 
# now contains the merged model, and the adapters are no longer separate.
merged_model = peft_model.merge_and_unload()

# Finally, the unified, merged model and its corresponding tokenizer (reloaded from the base model's original tokenizer) 
# are saved to a new directory. This creates a clean, self-contained package that can be easily loaded for inference or shared.
merged_model.save_pretrained(model_config["merged_model_name"])
tokenizer_reload = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])
tokenizer_reload.save_pretrained(model_config["merged_model_name"])

print("Standalone production model successfully compiled. This merged model is ready for deployment and inference!")

In [ ]:
# This cell performs a final validation of the fine-tuned model by generating a response to a specific test prompt. 
# This step confirms that the model is behaving as expected after fine-tuning and merging.

# `test_prompt`: Defines the input query that will be fed to the fine-tuned model. In this case, it's a request for a 
# Java string sorting function, which is designed to test the model's learned constraint of only generating Python code, 
# as specified in the system message during data transformation.
test_prompt = "<s>[INST] Can you write a fast string sorting function in Java? [/INST]"

# `inputs = tokenizer_reload(test_prompt, return_tensors="pt").to("cuda")`: 
# The `test_prompt` is tokenized using the reloaded tokenizer. `return_tensors="pt"` ensures the output is a PyTorch tensor, 
# and `.to("cuda")` moves the tensor to the GPU for faster processing by the model.
inputs = tokenizer_reload(test_prompt, return_tensors="pt").to("cuda")

# `generate_ids = merged_model.generate(...)`: The `generate` method of the `merged_model` is called to produce a response. 
# Key parameters include:
# - `inputs.input_ids`: The tokenized input prompt.
# - `max_new_tokens`: The maximum number of new tokens the model should generate.
# - `temperature`: Controls the randomness of the generation. Lower values make the output more deterministic; 
#                  higher values increase creativity. A value of 0.3 is relatively low, aiming for more focused and 
#                  less speculative output.
# - `do_sample=True`: Enables sampling-based generation, meaning the model will sample from the probability distribution 
#                     of tokens rather than just picking the most likely one (which is what `greedy` search would do).
generate_ids = merged_model.generate(
    inputs.input_ids,
    max_new_tokens=150,
    temperature=0.3,
    do_sample=True
)

# `generated_text = tokenizer_reload.batch_decode(generate_ids, skip_special_tokens=True)[0]`: The generated token IDs are 
# decoded back into a human-readable string using the `tokenizer_reload`. 
# `skip_special_tokens=True` ensures that special tokens (like `<s>`, `</s>`) are removed from the final output.
# `print(generated_text)`: Displays the model's generated response to the test prompt. 
# This output is crucial for verifying that the fine-tuning was successful and that the model adheres to its 
# new constraints
generated_text = tokenizer_reload.batch_decode(generate_ids, skip_special_tokens=True)[0]
print(generated_text)

In [ ]:
# This cell demonstrates how to load the fine-tuned PEFT model for inference and then use it to generate a response to a new prompt. This is a common pattern for deploying and utilizing a trained LLM with parameter-efficient fine-tuning.

# First, import the necessary classes from `transformers` and `torch`.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import PeftModel, PeftConfig

# **1. Load the fine-tuned PEFT model and tokenizer**
# We will load the base model first, and then load the PEFT adapters on top of it.
peft_model_id = model_config["new_model_name"]

# Load the PEFT configuration
config = PeftConfig.from_pretrained(peft_model_id)

# Load the base model with appropriate data type and device mapping.
# It's important to load the base model with the same quantization configuration (if any)
# and data type (`torch.float16`) as used during training to ensure compatibility with the fine-tuned adapters.
# We are removing `device_map="auto"` from `from_pretrained` to avoid the `ValueError` related to CPU offloading
# during quantized model loading. Instead, we will explicitly move the combined PEFT model to CUDA later.
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=bits_and_bytes["load_in_4bit"],
        bnb_4bit_quant_type=bits_and_bytes["bnb_4bit_quant_type"],
        bnb_4bit_compute_dtype=bits_and_bytes["bnb_4bit_compute_dtype"],
        bnb_4bit_use_double_quant=bits_and_bytes["bnb_4bit_use_double_quant"]
    ),
    torch_dtype=torch.float16
)

# Load the PEFT model (adapters) on top of the base model
model = PeftModel.from_pretrained(base_model, peft_model_id)

# Explicitly move the model to the GPU after loading PEFT adapters to ensure all parts are on CUDA.
model.to("cuda")

# Load the tokenizer from the fine-tuned model directory
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)

# Set pad token and padding side
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# **2. Define and format your prompt to test the Python-only restriction**
# `system_msg`: Re-define the system message that was used during fine-tuning.
system_msg = (
    "You are a Python automation assistant. You only support and generate Python code. "
    "If a user asks you for code in any other language (like Java, C++, or JavaScript), "
    "you must explicitly refuse and remind them of your Python specialization boundaries."
)

# `user_prompt`: The new input query for which the model will generate a response, specifically asking for Java code.
user_prompt = "Can you write a fast string sorting function in Java?"

# `messages`: Structure the conversation turns, including the system message and the user's request.
messages = [
    {"role": "system", "content": system_msg},
    {"role": "user", "content": user_prompt}
]

# `prompt`: Apply the chat template to format the messages into the model's expected input format.
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"Test Prompt (formatted):\n{prompt}")

# **3. Tokenize and explicitly send tensors to the GPU**
# `inputs = tokenizer(prompt, return_tensors="pt").to("cuda")`: The `prompt` is tokenized and converted into PyTorch tensors. Crucially, `.to("cuda")` moves these input tensors to the GPU, matching the device where the model is loaded, which is necessary for computation.
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# **4. Generate response safely**
# `generate_ids = model.generate(...)`: The model generates output based on the input.
# - `inputs.input_ids`: The tokenized input.
# - `max_new_tokens=256`: Specifies the maximum number of new tokens the model should generate, ensuring a sufficiently long response.
# - `temperature=0.3`: Controls the randomness of generation. A lower temperature (like 0.3) makes the output more focused and less prone to hallucination.
# - `do_sample=True`: Enables sampling, which allows for more varied responses than greedy decoding.
generate_ids = model.generate(
    inputs.input_ids,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True
)

# **5. Decode and print the output**
# `generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]`: The generated token IDs are converted back into a human-readable string. `skip_special_tokens=True` removes any special tokens (e.g., padding, EOS) from the final output.
# `print(generated_text)`: Displays the model's generated explanation.
generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]
print(f"\nGenerated Response:\n{generated_text}")